# Snowflake + Prefect: the next practical
**Same data, more pipeline features.** Run top to bottom in Colab. Restart the session before starting if Prefect was already imported.

| order_id | customer | amount (₹) |
|---|---|---:|
| 1 | Priya | 100 |
| 2 | Arjun | 200 |
| 3 | Meera | 150 |

We keep these values unchanged. Original sales remain **₹450**. For learning arithmetic transformations, we add a fictional **18% classroom tax**: ₹81 tax, ₹531 including tax. This is a teaching assumption, not a tax rule for actual sales.

You will learn extraction, safe repeat loading with MERGE, bronze/silver/gold layers, SQL calculations, validation, ranking, and Prefect task retries and logs.

| Layer | Table | Purpose |
|---|---|---|
| Bronze | orders | Keep order IDs, names and original amounts |
| Silver | orders | Standardize names; calculate tax and final amount |
| Gold | customer_sales | Total spending and order count per customer |
| Gold | sales_summary | Overall business totals |

This is **ELT**: Python extracts sample rows, loads them into Snowflake, and SQL transforms them there. Prefect coordinates the steps. It is a sequential teaching pipeline; run one flow at a time against this database.

## 1. Install
Restart Colab if requested.

In [ ]:
!pip install -q snowflake-connector-python "prefect>=3,<4" pandas

## 2. Connect to your Prefect dashboard
Enter your API key only in the hidden prompt. Your workspace URL is filled in from the earlier screenshot. Run this before importing Prefect.

Your Python runs in Colab. Prefect Cloud records states and logs. Printed sample data is included in those logs.


In [ ]:
import os
import re
from getpass import getpass

# Use the workspace's browser URL to locate its API.
workspace_url = "https://app.prefect.cloud/account/1db07649-353f-4e19-8021-644bee1b8296/workspace/d741b578-a867-4c10-b2b4-fb97fd4f7e80/home"
match = re.search(r"https://app\.prefect\.cloud/account/([^/?#]+)/workspace/([^/?#]+)", workspace_url)
if not match:
    raise ValueError("Open your workspace at app.prefect.cloud and copy its full browser URL.")

account_id, workspace_id = match.groups()
os.environ["PREFECT_API_URL"] = f"https://api.prefect.cloud/api/accounts/{account_id}/workspaces/{workspace_id}"
os.environ["PREFECT_API_KEY"] = getpass("Prefect API key (hidden): ").strip()
if not os.environ["PREFECT_API_KEY"]:
    raise ValueError("Enter your Prefect Cloud API key.")

# Configuration takes effect when Prefect is imported in the next section.
prefect_dashboard_url = f"https://app.prefect.cloud/account/{account_id}/workspace/{workspace_id}/dashboard"
print("Dashboard:", prefect_dashboard_url)
print("Cloud settings ready. Continue with the Snowflake connection cell.")

## 3. Connect to Snowflake
Use your existing account details below. Passwords are hidden prompts. A separate `NEXT_LAB_DB` keeps this practical separate from the first lab. The warehouse auto-suspends after 60 idle seconds; Snowflake usage can incur charges.

In [ ]:
from getpass import getpass
import snowflake.connector
from prefect import task, flow

# Your Snowflake account settings.
SNOWFLAKE_ACCOUNT = "sr43506.ap-southeast-7.aws"
SNOWFLAKE_USER = "SUMUNTHRA"
SNOWFLAKE_ROLE = "ACCOUNTADMIN"

# Ask for the password without displaying or saving it in this notebook.
SNOWFLAKE_PASSWORD = getpass("Snowflake password: ")

# Connect Python to Snowflake.
conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    role=SNOWFLAKE_ROLE,
)

# Confirm the connected user and role.
with conn.cursor() as cur:
    cur.execute("SELECT CURRENT_USER(), CURRENT_ROLE()")
    print("Connected:", cur.fetchone())


## 4. Create three layer schemas
A warehouse supplies compute. A database contains schemas. Each schema groups tables.

We create the bronze table once so its rows survive repeated runs. The silver and gold tables are rebuilt later from bronze.

In [ ]:
with conn.cursor() as cur:
    cur.execute("CREATE WAREHOUSE IF NOT EXISTS NEXT_LAB_WH WAREHOUSE_SIZE='XSMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE INITIALLY_SUSPENDED=TRUE")
    cur.execute("USE WAREHOUSE NEXT_LAB_WH")
    cur.execute("CREATE DATABASE IF NOT EXISTS NEXT_LAB_DB")
    cur.execute("USE DATABASE NEXT_LAB_DB")
    for schema in ["bronze", "silver", "gold"]:
        cur.execute(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS bronze.orders (
            order_id INTEGER,
            customer VARCHAR(30),
            amount NUMBER(10,2)
        )
    """)
print("Layers ready: bronze, silver, gold")

## 5. Input data and a small query helper
The input is exactly the same three orders. `read_rows` sends a SELECT and returns its result. Later, `display_query` shows those rows as a readable table.

In [ ]:
import pandas as pd
from IPython.display import display
from decimal import Decimal

ORDERS = [(1, "Priya", 100), (2, "Arjun", 200), (3, "Meera", 150)]

def read_rows(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall()

def display_query(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        frame = pd.DataFrame(cur.fetchall(), columns=[column[0] for column in cur.description])
    display(frame)
    return frame

## 6. Store the transformation SQL
**Silver:** `TRIM` removes surrounding spaces; `UPPER` standardizes names. `ROUND` rounds arithmetic to two decimals. Base amount stays unchanged.

**Gold:** `GROUP BY` collects each customer's orders. `COUNT` counts orders; `SUM` adds money; `AVG` finds average order value. Each SQL statement builds one table.

In [ ]:
SQL_SILVER = """
-- Standardize names and add calculated columns.
CREATE OR REPLACE TABLE silver.orders AS
SELECT order_id,
       UPPER(TRIM(customer)) AS customer,
       amount,
       ROUND(amount * 0.18, 2) AS tax_amount,
       amount + ROUND(amount * 0.18, 2) AS amount_with_tax,
       CASE WHEN amount >= 150 THEN 'HIGH' ELSE 'LOW' END AS order_category
FROM bronze.orders
"""

SQL_CUSTOMER_SALES = """
-- Summarize orders separately for each customer.
CREATE OR REPLACE TABLE gold.customer_sales AS
SELECT customer,
       COUNT(*) AS order_count,
       SUM(amount) AS base_sales,
       SUM(tax_amount) AS tax_total,
       SUM(amount_with_tax) AS sales_with_tax
FROM silver.orders
GROUP BY customer
"""

SQL_SUMMARY = """
-- Produce a single row of overall business metrics.
CREATE OR REPLACE TABLE gold.sales_summary AS
SELECT COUNT(*) AS total_orders,
       SUM(amount) AS base_sales,
       AVG(amount) AS average_order,
       SUM(tax_amount) AS tax_total,
       SUM(amount_with_tax) AS sales_with_tax
FROM silver.orders
"""

## 7. Prefect tasks
`@task` records a step's execution. `retries=2` allows up to **three attempts**: the initial attempt plus two retries.

MERGE matches `order_id`. If the row exists, it updates it; otherwise it inserts it. Loading the same input again does not add extra orders. A transaction groups the three merges so they commit together. Retrying the transformation safely rebuilds its table.

The input validation checks our required fields and unique IDs before loading. Snowflake's ordinary tables do not enforce uniqueness just because an ID column exists. MERGE here assumes one input row per ID and a clean, dedicated bronze table; avoid concurrent runs.


In [ ]:
@task(name="Extract and check input")
def extract_orders():
    # EXTRACT: read the small source; check values before loading.
    rows = list(ORDERS)
    if len({row[0] for row in rows}) != len(rows):
        raise ValueError("Source has duplicate order IDs")
    for order_id, customer, amount in rows:
        if order_id is None or not isinstance(customer, str) or not customer.strip():
            raise ValueError("Order ID and customer are required")
        if amount is None or amount < 0:
            raise ValueError("Amount must be present and non-negative")
    print("Extracted", len(rows), "orders")
    return rows


@task(name="Load bronze with MERGE", retries=2, retry_delay_seconds=3)
def load_orders(rows):
    # LOAD: update existing IDs or insert new ones; do not append duplicates.
    with conn.cursor() as cur:
        cur.execute("BEGIN")
        try:
            for row in rows:
                cur.execute("""
                    MERGE INTO bronze.orders AS target
                    USING (SELECT %s AS order_id, %s AS customer, %s AS amount) AS source
                    ON target.order_id = source.order_id
                    WHEN MATCHED THEN UPDATE SET
                        customer = source.customer, amount = source.amount
                    WHEN NOT MATCHED THEN INSERT (order_id, customer, amount)
                        VALUES (source.order_id, source.customer, source.amount)
                """, row)
            cur.execute("COMMIT")
        except Exception:
            cur.execute("ROLLBACK")
            raise
    print("Merged", len(rows), "orders into bronze")


@task(task_run_name="{step}", retries=2, retry_delay_seconds=3)
def transform_table(step, sql):
    # TRANSFORM: Snowflake runs one SQL statement per task.
    with conn.cursor() as cur:
        cur.execute(sql)
    print(step, "completed")


@task(name="Validate the result")
def validate_results(expected_rows):
    # Compare IDs, names and amounts with the source; do not rely on counts alone.
    expected = sorted((i, name.strip().upper(), Decimal(str(amount)))
                      for i, name, amount in expected_rows)
    actual = read_rows("SELECT order_id, customer, amount FROM silver.orders ORDER BY order_id")
    if actual != expected:
        raise ValueError(f"Silver data differs from source: {actual}")

    base = sum((row[2] for row in expected), Decimal("0"))
    tax = sum((round(row[2] * Decimal("0.18"), 2) for row in expected), Decimal("0"))
    summary = read_rows("SELECT total_orders, base_sales, tax_total, sales_with_tax FROM gold.sales_summary")[0]
    if tuple(summary) != (len(expected), base, tax, base + tax):
        raise ValueError(f"Incorrect gold summary: {summary}")
    customer_total = read_rows("SELECT SUM(base_sales) FROM gold.customer_sales")[0][0]
    if customer_total != base:
        raise ValueError("Customer totals do not reconcile")
    print(f"Validation passed: {len(expected)} orders; base ₹{base}; tax ₹{tax}; final ₹{base+tax}")
    return {"orders": len(expected), "base_sales": float(base),
            "tax": float(tax), "sales_with_tax": float(base + tax)}


@task(name="Log the sales report")
def report_sales(metrics):
    # Display meaningful output in Colab and in Prefect Cloud logs.
    print("CUSTOMER SALES")
    for row in read_rows("SELECT customer, base_sales, tax_total, sales_with_tax FROM gold.customer_sales ORDER BY base_sales DESC"):
        print(row)
    print("Overall metrics:", metrics)
    return metrics

## 8. Run the flow
The flow states the dependencies in plain Python: extract → load → silver → gold → validate → report.

If a task fails after its retries, downstream calls are not reached. Deterministic input errors have no retries because repeating the same invalid data will not fix them.

In [ ]:
@flow(name="snowflake-orders-next-lab", log_prints=True)
def orders_pipeline():
    rows = extract_orders()
    load_orders(rows)
    transform_table("Build silver orders", SQL_SILVER)
    transform_table("Build customer sales", SQL_CUSTOMER_SALES)
    transform_table("Build overall summary", SQL_SUMMARY)
    metrics = validate_results(rows)
    return report_sales(metrics)

result = orders_pipeline()
print(result)

## 9. Inspect the Snowflake tables
Expected: three orders in bronze and silver; three customer rows in gold; one overall summary row.

| Customer | Base sales | Tax | Including tax |
|---|---:|---:|---:|
| PRIYA | 100 | 18 | 118 |
| ARJUN | 200 | 36 | 236 |
| MEERA | 150 | 27 | 177 |
| Total | 450 | 81 | 531 |


In [ ]:
display_query("SELECT * FROM bronze.orders ORDER BY order_id")
display_query("SELECT * FROM silver.orders ORDER BY order_id")
display_query("SELECT * FROM gold.customer_sales ORDER BY base_sales DESC")
display_query("SELECT * FROM gold.sales_summary")

## 10. More complex queries on the same data
- `WHERE` filters order rows before aggregation.
- `HAVING` filters customer groups after aggregation.
- `DENSE_RANK` assigns rank 1 to the highest amount; ties share a rank.
- Dividing by total sales calculates each customer's contribution.

Each customer currently has one order, so customer totals equal individual order amounts. Adding a second order for Priya later will make the grouping more apparent.

In [ ]:
# Find high-value orders: Arjun and Meera.
display_query("SELECT customer, amount FROM silver.orders WHERE amount >= 150 ORDER BY amount DESC")

# Find customers whose total spending exceeds ₹100: Arjun and Meera.
display_query("""
    SELECT customer, SUM(amount) AS spending
    FROM silver.orders
    GROUP BY customer
    HAVING SUM(amount) > 100
    ORDER BY spending DESC
""")

# Rank customers: Arjun=1, Meera=2, Priya=3.
display_query("""
    SELECT customer, base_sales,
           DENSE_RANK() OVER (ORDER BY base_sales DESC) AS spending_rank
    FROM gold.customer_sales
    ORDER BY spending_rank, customer
""")

# Calculate contribution to base sales: 44.44%, 33.33%, 22.22%.
display_query("""
    SELECT customer, base_sales,
           ROUND(100.0 * base_sales /
                 NULLIF((SELECT SUM(base_sales) FROM gold.customer_sales), 0), 2) AS sales_percent
    FROM gold.customer_sales
    ORDER BY base_sales DESC
""")

## 11. Repeat the pipeline
MERGE prevents the same input from creating duplicate orders. Bronze keeps IDs not present in a later input: this is an upsert example, not full synchronization or deletion handling. The sample validation assumes the dedicated database contains only our three source orders.

In [ ]:
second_result = orders_pipeline()
assert second_result == result
print("Same totals after the second run:", second_result)

## 12. Where to see the output
**Prefect:** open the printed dashboard URL → Runs → latest run of `snowflake-orders-next-lab`. Inspect the seven task runs, Completed/Failed states, timings and printed logs. Successful input does not trigger retries; the retry policy applies only when a load or transformation fails.

**Snowflake:** open Projects → a SQL worksheet, then run:

```sql
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE NEXT_LAB_WH;
SELECT * FROM NEXT_LAB_DB.BRONZE.ORDERS ORDER BY order_id;
SELECT * FROM NEXT_LAB_DB.SILVER.ORDERS ORDER BY order_id;
SELECT * FROM NEXT_LAB_DB.GOLD.CUSTOMER_SALES ORDER BY base_sales DESC;
SELECT * FROM NEXT_LAB_DB.GOLD.SALES_SUMMARY;
```

**Student questions:** Why does the second run still contain three orders? Why is tax a separate column? How are WHERE and HAVING different? Which task would fail if a source amount became negative?

This lab does not schedule or deploy a job. It does not guarantee recovery from connection loss; persistent connection failures may require reconnecting and rerunning. Tables are rebuilt individually, so the entire flow is not one atomic database transaction.

References:
- https://docs.snowflake.com/en/sql-reference/sql/merge
- https://docs.prefect.io/v3/how-to-guides/workflows/retries
- https://docs.prefect.io/v3/how-to-guides/cloud/connect-to-cloud


In [ ]:
# Close only after completing the practical.
# conn.close()